In [1]:
from pathlib import Path

import pandas as pd

In [2]:
TENANT_INPUT  = Path("../../data/headlines/tenants_title.csv")
LANDLORD_INPUT = Path("../../data/headlines/landlords_title.csv")

OUTPUT_DIR     = Path("../../data/headlines_all")
COVERAGE_FILE  = OUTPUT_DIR / "coverage_pivot.csv"
TENANT_OUTPUT  = OUTPUT_DIR / "headlines_tenant.csv"
LANDLORD_OUTPUT = OUTPUT_DIR / "headlines_landlord.csv"
STATS_OUTPUT   = OUTPUT_DIR / "headline_stats.csv"

MIN_COVERAGE = 100   # min English headlines per source-year to include
START_YEAR   = 2015
END_YEAR     = 2025

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Canadian Housing Headlines Dataset

Processes two headline datasets downloaded from the MediaCloud **Canada - National** collection (ID: 34411583), 2015–2025:

- **`tenants_title.csv`** — titles containing: `tenant`, `tenants`, `tenancy`, `renter`, `renters`
- **`landlords_title.csv`** — titles containing: `landlord`, `landlords`

**Pipeline:**
1. Load raw downloaded files → add `year` column from `publish_date`
2. **Language filter** — keep `language == 'en'` or null
3. **Coverage filter** — keep only source × year pairs with ≥ 100 English headlines (per `coverage_pivot.csv`)
4. **Duplicate tagging** — wire-service stories (same title + publish date) tagged `is_duplicate = True`
5. Save to `data/headlines_all/`

Deduplication by source hierarchy is in Section 6 (fill in after reviewing Section 5).

## 1. Load Data

Load the raw downloaded CSV files and derive a `year` column from `publish_date`. Also load `coverage_pivot.csv` to build the set of valid `(source, year)` pairs used by the coverage filter.

In [3]:
coverage_pivot = pd.read_csv(COVERAGE_FILE, index_col="source")
print(f"Coverage pivot: {coverage_pivot.shape[0]} sources × {coverage_pivot.shape[1]} years")

valid_source_years = set()
for source in coverage_pivot.index:
    for year in range(START_YEAR, END_YEAR + 1):
        col = str(year)
        if col in coverage_pivot.columns and coverage_pivot.loc[source, col] >= MIN_COVERAGE:
            valid_source_years.add((source, year))

valid_sources = {s for s, _ in valid_source_years}
print(f"Valid source-year pairs (≥ {MIN_COVERAGE} headlines): {len(valid_source_years)}")
print(f"Unique sources: {len(valid_sources)}")

Coverage pivot: 108 sources × 11 years
Valid source-year pairs (≥ 100 headlines): 656
Unique sources: 88


In [4]:
tenant_raw = pd.read_csv(TENANT_INPUT)
tenant_raw["year"] = pd.to_datetime(tenant_raw["publish_date"]).dt.year
print(f"Tenant   raw: {len(tenant_raw):,} stories  |  years {tenant_raw['year'].min()}–{tenant_raw['year'].max()}")

landlord_raw = pd.read_csv(LANDLORD_INPUT)
landlord_raw["year"] = pd.to_datetime(landlord_raw["publish_date"]).dt.year
print(f"Landlord raw: {len(landlord_raw):,} stories  |  years {landlord_raw['year'].min()}–{landlord_raw['year'].max()}")

Tenant   raw: 5,684 stories  |  years 2015–2025
Landlord raw: 3,140 stories  |  years 2015–2025


## 2. Filter

1. **Language** — keep `language == 'en'` or null.
2. **Coverage** — keep `(media_name, year)` pairs in `valid_source_years` (≥ `MIN_COVERAGE` English headlines).

In [5]:
def filter_language(df, label=""):
    if df.empty or "language" not in df.columns:
        return df
    mask = df["language"].isin(["en"]) | df["language"].isna()
    filtered = df[mask].copy()
    print(f"{label}: {(~mask).sum():,} non-English removed → {len(filtered):,} remain")
    return filtered


tenant_df   = filter_language(tenant_raw.copy(),   "Tenant")
landlord_df = filter_language(landlord_raw.copy(), "Landlord")

Tenant: 0 non-English removed → 5,684 remain
Landlord: 0 non-English removed → 3,140 remain


In [6]:
def filter_by_coverage(df, valid_pairs, source_col="media_name", label=""):
    if df.empty:
        return df
    if source_col not in df.columns:
        print(f"WARNING: '{source_col}' not found. Available: {list(df.columns)}")
        return df
    mask = pd.Series(
        [k in valid_pairs for k in zip(df[source_col], df["year"])],
        index=df.index,
    )
    filtered = df[mask].copy()
    print(f"{label}: {(~mask).sum():,} stories from under-threshold source-years removed → {len(filtered):,} remain")
    return filtered


tenant_df   = filter_by_coverage(tenant_df,   valid_source_years, label="Tenant")
landlord_df = filter_by_coverage(landlord_df, valid_source_years, label="Landlord")

Tenant: 1 stories from under-threshold source-years removed → 5,683 remain
Landlord: 0 stories from under-threshold source-years removed → 3,140 remain


## 3. Duplicate Detection

Wire-service stories appear across multiple outlets with the same title on the same date. All stories sharing an identical `(title, publish_date)` pair (case-insensitive) are tagged `is_duplicate = True`. No rows are removed here; deduplication is in Section 6.

In [7]:
def tag_duplicates(df, label=""):
    if df.empty:
        return df
    df = df.copy()
    title_norm = df["title"].fillna("").str.strip().str.lower()
    date_str = df["publish_date"].astype(str)
    dup_key = title_norm + "|||" + date_str
    df["is_duplicate"] = dup_key.duplicated(keep=False)
    n_dup = df["is_duplicate"].sum()
    n_groups = dup_key[df["is_duplicate"]].nunique()
    print(f"{label}: {n_dup:,} stories flagged as duplicates across {n_groups:,} unique title+date groups")
    return df


tenant_df = tag_duplicates(tenant_df, "Tenant")
landlord_df = tag_duplicates(landlord_df, "Landlord")

Tenant: 1,485 stories flagged as duplicates across 431 unique title+date groups
Landlord: 954 stories flagged as duplicates across 270 unique title+date groups


## 4. Save

Select canonical columns and write to `data/headlines_all/`. Both files include `is_duplicate` but no stories are removed.

In [8]:
KEEP_COLS = ["id", "title", "url", "publish_date", "language", "media_name", "media_url", "year", "is_duplicate"]


def save_headlines(df, path, label=""):
    if df.empty:
        print(f"{label}: empty — skipped.")
        return
    cols = [c for c in KEEP_COLS if c in df.columns]
    missing = [c for c in KEEP_COLS if c not in df.columns]
    if missing:
        print(f"  {label}: columns missing (omitted): {missing}")
    df[cols].to_csv(path, index=False)
    print(f"{label}: {len(df):,} rows → {path}")


save_headlines(tenant_df,   TENANT_OUTPUT,   "Tenant")
save_headlines(landlord_df, LANDLORD_OUTPUT, "Landlord")

Tenant: 5,683 rows → ../../data/headlines_all/headlines_tenant.csv
Landlord: 3,140 rows → ../../data/headlines_all/headlines_landlord.csv


## 5. Summary Statistics

Yearly breakdown: story counts, duplicate counts, unique sources, and duplicate rate per category.

In [9]:
def yearly_stats(df, category):
    if df.empty:
        return pd.DataFrame()
    return (
        df.groupby("year")
        .agg(
            n_stories=("id", "count"),
            n_duplicates=("is_duplicate", "sum"),
            n_sources=("media_name", "nunique"),
        )
        .assign(
            pct_duplicate=lambda x: (100 * x["n_duplicates"] / x["n_stories"]).round(1),
            category=category,
        )
    )


stats = pd.concat([yearly_stats(tenant_df, "tenant"), yearly_stats(landlord_df, "landlord")])
stats.to_csv(STATS_OUTPUT)
print(stats.to_string())

      n_stories  n_duplicates  n_sources  pct_duplicate  category
year                                                             
2015        245            34         13           13.9    tenant
2016        299            37         21           12.4    tenant
2017        518            93         42           18.0    tenant
2018        575           115         43           20.0    tenant
2019        569            57         48           10.0    tenant
2020        723           160         51           22.1    tenant
2021        464            97         43           20.9    tenant
2022        474            73         41           15.4    tenant
2023        473           103         37           21.8    tenant
2024        821           414         28           50.4    tenant
2025        522           302         26           57.9    tenant
2015        150            12          9            8.0  landlord
2016        177            18         21           10.2  landlord
2017      

## 6. Duplicate Analysis

Report every source with at least one duplicate story. Provide a ranked source hierarchy (highest-priority first) to fill in Section 7.

In [10]:
def sources_with_duplicates(df, label):
    if df.empty or "is_duplicate" not in df.columns:
        return pd.DataFrame()
    dup_df = df[df["is_duplicate"]]
    if dup_df.empty:
        print(f"{label}: no duplicates found.")
        return pd.DataFrame()
    summary = (
        dup_df.groupby("media_name")
        .agg(n_duplicate_stories=("id", "count"))
        .sort_values("n_duplicate_stories", ascending=False)
    )
    print(f"{label.upper()} — {len(summary)} sources with duplicates:")
    print(summary.to_string())
    return summary


tenant_dup_sources   = sources_with_duplicates(tenant_df,   "tenant")
landlord_dup_sources = sources_with_duplicates(landlord_df, "landlord")

all_dup_sources = set()
if not tenant_dup_sources.empty:
    all_dup_sources |= set(tenant_dup_sources.index)
if not landlord_dup_sources.empty:
    all_dup_sources |= set(landlord_dup_sources.index)

print(f"\nAll sources with any duplicates ({len(all_dup_sources)} total):")
for s in sorted(all_dup_sources):
    print(f"  {s}")

TENANT — 42 sources with duplicates:
                             n_duplicate_stories
media_name                                      
wellandtribune.ca                            137
niagarafallsreview.ca                        134
thestar.com                                  127
thespec.com                                  114
therecord.com                                101
winnipegfreepress.com                         77
globalnews.ca                                 66
brandonsun.com                                62
cbc.ca                                        58
ctvnews.ca                                    57
pentictonherald.ca                            55
kelownadailycourier.ca                        55
timescolonist.com                             53
abbynews.com                                  46
trailtimes.ca                                 45
kimberleybulletin.com                         43
nationalnewswatch.com                         41
lethbridgeherald.com            

## 7. Deduplication

Keeps one story per duplicate group (same title + publish date), favouring the highest-ranked source in the hierarchy. Overwrites the output CSVs.

In [11]:
DEDUP_HIERARCHY = [
    # Tier 1 — National broadcasters
    "cbc.ca",
    "ctvnews.ca",
    "globalnews.ca",
    # Tier 2 — National print
    "thestar.com",
    "theglobeandmail.com",
    "nationalpost.com",
    "nationalobserver.com",
    # Tier 3 — Major regional papers
    "vancouversun.com",
    "ottawacitizen.com",
    "montrealgazette.com",
    "edmontonjournal.com",
    "calgaryherald.com",
    "winnipegfreepress.com",
    "timescolonist.com",
    # Tier 4 — Mid-size metros
    "thespec.com",
    "lfpress.com",
    "therecord.com",
    "torontosun.com",
    "calgarysun.com",
    "edmontonsun.com",
    "winnipegsun.com",
    "thewhig.com",
    "intelligencer.ca",
    # Tier 5 — Smaller regionals
    "brandonsun.com",
    "brantfordexpositor.ca",
    "niagarafallsreview.ca",
    "wellandtribune.ca",
    "lethbridgeherald.com",
    "reddeeradvocate.com",
    "kelownadailycourier.ca",
    "pentictonherald.ca",
    "abbynews.com",
    "medicinehatnews.com",
    "trailtimes.ca",
    "kimberleybulletin.com",
    "dailyheraldtribune.com",
    "nugget.ca",
    "tj.news",
    "standard-freeholder.com",
    "woodstocksentinelreview.com",
    "bclocalnews.com",
    # Tier 6 — Aggregators / specialty
    "canadianbusiness.com",
    "moneysense.ca",
    "canoe.com",
    "nationalnewswatch.com",
    "canadanews.net",
    "canadastandard.com",
]


def deduplicate_by_hierarchy(df, hierarchy):
    rank = {src: i for i, src in enumerate(hierarchy)}
    df = df.copy()
    df["_rank"] = df["media_name"].map(rank).fillna(len(hierarchy))
    df["_dup_key"] = df["title"].fillna("").str.strip().str.lower() + "|||" + df["publish_date"].astype(str)
    best_idx = df.groupby("_dup_key")["_rank"].idxmin()
    deduped = df.loc[best_idx].drop(columns=["_rank", "_dup_key"])
    print(f"Removed {len(df) - len(deduped):,} duplicates; {len(deduped):,} stories remain.")
    return deduped


tenant_deduped   = deduplicate_by_hierarchy(tenant_df,   DEDUP_HIERARCHY)
landlord_deduped = deduplicate_by_hierarchy(landlord_df, DEDUP_HIERARCHY)

save_headlines(tenant_deduped,   TENANT_OUTPUT,   "Tenant (deduped)")
save_headlines(landlord_deduped, LANDLORD_OUTPUT, "Landlord (deduped)")

Removed 1,054 duplicates; 4,629 stories remain.
Removed 684 duplicates; 2,456 stories remain.
Tenant (deduped): 4,629 rows → ../../data/headlines_all/headlines_tenant.csv
Landlord (deduped): 2,456 rows → ../../data/headlines_all/headlines_landlord.csv
